**Import intersection definition from mycity**

In the folder intersections there are files from Landau intersections

In [1]:
folder = 'input/intersections'

In [2]:
import os
import json

def find_json_files(directory):
    files = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        files.append(filepath)
    return files

In [3]:

json_files = find_json_files(folder)

**Interpret Contents of the files**

**Parse first file and understand the content**

In [4]:
import json
from collections import defaultdict

for file in json_files:

    with open(file, 'r') as f:  
        json_data = json.load(f)  

    # Extract the intersection name
    intersection_name = json_data["tlc"]["name"]

    if intersection_name == 'LD-LSA16':
    
        # Extract the detector IDs
        detector_ids = []
        print(f"### Intersection Name: {intersection_name}")

        for sw_status in json_data.get("tlc", {}).get("softwareStatus", []):
            node = sw_status.get("node", {})
            detectors = node.get("detectors", [])

            for d in detectors:
                print(
                    f"Detector ID={d['id']}, "
                    f"Name={d['name']}, "
                    f"Type={d['detectorType']}"
                )

### Intersection Name: LD-LSA16
Detector ID=1, Name=VD11, Type=NONE
Detector ID=2, Name=VD12, Type=NONE
Detector ID=3, Name=WD21, Type=NONE
Detector ID=4, Name=WD22, Type=NONE
Detector ID=5, Name=VD23, Type=NONE
Detector ID=6, Name=VD31, Type=NONE
Detector ID=7, Name=VD32, Type=NONE
Detector ID=8, Name=WD41, Type=NONE
Detector ID=9, Name=WD42, Type=NONE
Detector ID=10, Name=VD43, Type=NONE
Detector ID=11, Name=VD44, Type=NONE
Detector ID=12, Name=D24, Type=NONE
Detector ID=13, Name=D25, Type=NONE
Detector ID=14, Name=D45, Type=NONE
Detector ID=15, Name=D46, Type=NONE
Detector ID=16, Name=DK1_s, Type=NONE
Detector ID=17, Name=DK3_s, Type=NONE
Detector ID=18, Name=DK1_b, Type=NONE
Detector ID=19, Name=DK2_b, Type=NONE
Detector ID=20, Name=DK3_b, Type=NONE
Detector ID=21, Name=DK4_b, Type=NONE
Detector ID=22, Name=WBK1_Err, Type=NONE
Detector ID=23, Name=WBK2_Err, Type=NONE
Detector ID=24, Name=VK14_i_O, Type=NONE
Detector ID=25, Name=PA, Type=NONE
Detector ID=26, Name=PB, Type=NONE
Detec

This code loads the LD-LSA16 intersection from MyCity JSON files and filters only the selected vehicle detectors that will be used for calibration and LSTM input.


In [5]:
import json

# Kullanmak istediğin detector isimleri (JSON yazımıyla)
selected_detectors = {
    "VD11",  # VD1.1  south right
    "VD12",  # VD1.2  south through
    "VD13",  # VD1.3  south through
    "VD23",  # VD2.3  west through+right
    "WD22",  # WD2.2  west left
    "D24",   # D2.4   west upstream (65m)
    "VD31",  # VD3.1  north right
    "VD32",  # VD3.2  north through
    "VD43",  # VD4.3  east through+right
    "VD44"   # VD4.4  east left
}

target_intersection = "LD-LSA16"

for file in json_files:
    with open(file, "r") as f:
        json_data = json.load(f)

    intersection_name = json_data.get("tlc", {}).get("name")

    if intersection_name != target_intersection:
        continue

    print(f"\nFiltered detectors for {intersection_name}:\n")

    for sw_status in json_data.get("tlc", {}).get("softwareStatus", []):
        for d in sw_status.get("node", {}).get("detectors", []):
            if d.get("name") in selected_detectors:
                print(
                    f"ID={d.get('id')} | "
                    f"Name={d.get('name')} | "
                    f"Type={d.get('detectorType')}"
                )



Filtered detectors for LD-LSA16:

ID=1 | Name=VD11 | Type=NONE
ID=2 | Name=VD12 | Type=NONE
ID=4 | Name=WD22 | Type=NONE
ID=5 | Name=VD23 | Type=NONE
ID=6 | Name=VD31 | Type=NONE
ID=7 | Name=VD32 | Type=NONE
ID=10 | Name=VD43 | Type=NONE
ID=11 | Name=VD44 | Type=NONE
ID=12 | Name=D24 | Type=NONE
ID=33 | Name=VD13 | Type=NONE


**Check state before using AI**

In [10]:
import json

def check_state_extended(tlc: dict):
    reasons = []
    zero_detectors = []

    # 1) Active?
    if tlc.get("active") is False:
        reasons.append("active=False")

    # 2) General State?
    state = tlc.get("state")
    if state is not None and state != "NO_FAILURE":
        reasons.append(f"state={state} (expected NO_FAILURE)")

    # 3) Broken detector count?
    broken = tlc.get("brokenDetectorCount")
    if isinstance(broken, int) and broken > 0:
        reasons.append(f"brokenDetectorCount={broken}")

    # 4) Hardware status
    hw = tlc.get("hardwareStatus", {})
    if isinstance(hw, dict):
        if hw.get("powerOk") is False:
            reasons.append("hardwareStatus.powerOk=False")
        if hw.get("emergencyOff") is True:
            reasons.append("hardwareStatus.emergencyOff=True")
        if hw.get("doorOpen") is True:
            reasons.append("hardwareStatus.doorOpen=True")
        if hw.get("persistenceOk") is False:
            reasons.append("hardwareStatus.persistenceOk=False")

    # 5) Detector reading check (always zero)
    for sw in tlc.get("softwareStatus", []):
        node = sw.get("node", {})
        for d in node.get("detectors", []):
            # bazı exportlarda count/value alanı olur
            value = d.get("value") or d.get("count")
            if isinstance(value, (int, float)) and value == 0:
                zero_detectors.append(d.get("name"))

    ok = (len(reasons) == 0)
    return ok, reasons, zero_detectors


# Run for LD-LSA16
for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        json_data = json.load(f)

    tlc = json_data.get("tlc", {})
    intersection_name = tlc.get("name")

    if intersection_name == "LD-LSA16":
        ok, reasons, zero_detectors = check_state_extended(tlc)

        print(f"### STATE CHECK: {intersection_name}")

        if ok:
            print("OK -> AI/analysis can proceed")
        else:
            print("NOT OK -> do NOT use AI on this snapshot")
            for r in reasons:
                print(" -", r)

        # Fault detectors (controller tarafindan bildirilen)
        fault = tlc.get("faultDetectors", [])
        if fault:
            print("\nController-reported faulty detectors:")
            for fd in fault:
                print(f" - ID={fd.get('id')} | Name={fd.get('name')}")

        # Sürekli 0 veren dedektörler
        if zero_detectors:
            print("\nPotential always-zero detectors:")
            for z in zero_detectors:
                print(" -", z)

        print("\nSummary:",
              "state=", tlc.get("state"),
              "| active=", tlc.get("active"),
              "| brokenDetectorCount=", tlc.get("brokenDetectorCount"),
              "| powerOk=", tlc.get("hardwareStatus", {}).get("powerOk"),
              "| emergencyOff=", tlc.get("hardwareStatus", {}).get("emergencyOff"))
        break


### STATE CHECK: LD-LSA16
NOT OK -> do NOT use AI on this snapshot
 - state=DETECTOR_FAILURE (expected NO_FAILURE)
 - brokenDetectorCount=2

Summary: state= DETECTOR_FAILURE | active= True | brokenDetectorCount= 2 | powerOk= True | emergencyOff= False


**Which dedectors have failure?**

In [9]:
for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        j = json.load(f)
#which dedectors have failure?
    tlc = j.get("tlc", {})
    if tlc.get("name") == "LD-LSA16":
        hw = tlc.get("hardwareStatus", {})
        print("faultDetectors:", hw.get("faultDetectors"))
        break


faultDetectors: [{'detectorType': 'NONE', 'id': '8', 'name': 'WD41'}, {'detectorType': 'NONE', 'id': '9', 'name': 'WD42'}]


This script analyzes the 15-minute time-series detector data of LD-LSA16 and identifies detectors that constantly or almost constantly report zero values. It computes the percentage of zero readings for count, occupancy, and velocity across the full analysis period to detect faulty or inactive sensors before using the data for calibration or AI modeling.


In [ ]:
import os
import json
from collections import defaultdict

# 1) Intersection definition path 
intersections_folder = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"

# 2) Time-series detector readings path 
landau_folder = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"
lsa = "LD-LSA16"
lsa_timeseries_folder = os.path.join(landau_folder, lsa)

# The detectors that I want to use
selected_names = {"VD23", "WD22", "D24", "VD11", "VD12", "VD13", "VD43", "VD44", "VD31", "VD32"}

def find_json_files(directory):
    files = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if os.path.isfile(filepath) and filename.lower().endswith(".json"):
            files.append(filepath)
    return files

# ---------- A) Intersection definition ID <-> Name map  ----------
def load_detector_id_name_map(intersections_folder, alias_target):
    for fp in find_json_files(intersections_folder):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        tlc = data.get("tlc", {})
        if tlc.get("alias") != alias_target:
            continue

        id_to_name = {}
        name_to_id = {}

        for sw in tlc.get("softwareStatus", []):
            node = sw.get("node", {})
            for d in node.get("detectors", []):
                d_id = d.get("id")
                d_name = d.get("name")
                if d_id is not None and d_name:
                    # id bazen string gelebiliyor, normalize edelim
                    d_id_int = int(d_id)
                    id_to_name[d_id_int] = d_name
                    name_to_id[d_name] = d_id_int

        return id_to_name, name_to_id

    return {}, {}

id_to_name, name_to_id = load_detector_id_name_map(intersections_folder, lsa)

if not id_to_name:
    print(f"WARNING: Could not find detector map for {lsa} in intersections folder.")

# ---------- B) Time-series dosyalarını oku ve zero analizi yap ----------
def safe_get_reading(det_obj, key):
    """
    reading içinden count/occupancy/velocity değerini çeker.
    beklenen: det["reading"][key]["value"]
    """
    reading = det_obj.get("reading", {})
    box = reading.get(key, {})
    val = box.get("value")
    return val

stats = defaultdict(lambda: {
    "n_frames": 0,
    "count_zero": 0, "count_nonzero": 0,
    "occ_zero": 0, "occ_nonzero": 0,
    "vel_zero": 0, "vel_nonzero": 0
})

files = find_json_files(lsa_timeseries_folder)
files_sorted = sorted(files)  # tarih sırasına yakın

for fp in files_sorted:
    with open(fp, "r", encoding="utf-8") as f:
        day = json.load(f)

    for tf in day.get("timeFrames", []):
        detectors = tf.get("detectors", [])
        for det in detectors:
            det_id = det.get("id")
            if det_id is None:
                continue
            det_id = int(det_id)

            # frame say
            stats[det_id]["n_frames"] += 1

            c = safe_get_reading(det, "count")
            o = safe_get_reading(det, "occupancy")
            v = safe_get_reading(det, "velocity")

            if isinstance(c, (int, float)):
                if c == 0:
                    stats[det_id]["count_zero"] += 1
                else:
                    stats[det_id]["count_nonzero"] += 1

            if isinstance(o, (int, float)):
                if o == 0:
                    stats[det_id]["occ_zero"] += 1
                else:
                    stats[det_id]["occ_nonzero"] += 1

            if isinstance(v, (int, float)):
                if v == 0:
                    stats[det_id]["vel_zero"] += 1
                else:
                    stats[det_id]["vel_nonzero"] += 1

# ---------- C) Rapor bas ----------
def pct(z, n):
    return 0.0 if n == 0 else (100.0 * z / n)

# Eğer sadece seçtiğin dedektörleri raporlamak istersen:
selected_ids = {name_to_id[n] for n in selected_names if n in name_to_id}

print(f"\n### CONSTANT-ZERO CHECK (time-series) for {lsa}")
print(f"Daily files read: {len(files_sorted)}")
print(f"Selected detector names: {sorted(selected_names)}")
print(f"Selected detector ids found: {sorted(selected_ids)}")

# 1) Seçili dedektörler için detay
print("\n=== Selected detectors: usability indicators ===")
for det_id in sorted(selected_ids):
    n = stats[det_id]["n_frames"]
    name = id_to_name.get(det_id, f"(id={det_id})")

    cz = stats[det_id]["count_zero"]
    oz = stats[det_id]["occ_zero"]
    vz = stats[det_id]["vel_zero"]

    print(
        f"- ID={det_id} | Name={name} | frames={n} | "
        f"count_zero={cz} ({pct(cz,n):.1f}%) | "
        f"occ_zero={oz} ({pct(oz,n):.1f}%) | "
        f"vel_zero={vz} ({pct(vz,n):.1f}%)"
    )

# 2) Tüm dedektörler içinde %100 count=0 olanlar
print("\n=== CONSTANTLY ZERO (count) across all frames ===")
any_constant = False
for det_id in sorted(stats.keys()):
    n = stats[det_id]["n_frames"]
    if n == 0:
        continue
    if stats[det_id]["count_zero"] == n and stats[det_id]["count_nonzero"] == 0:
        any_constant = True
        name = id_to_name.get(det_id, f"(id={det_id})")
        print(f"- ID={det_id} | Name={name} | frames={n} | count is ALWAYS 0")
if not any_constant:
    print("(none)")

# 3) İstersen threshold ile (örn %99+ zero) “almost always zero” da çıkar:
threshold = 99.0
print(f"\n=== ALMOST ALWAYS ZERO (count >= {threshold}%) ===")
any_almost = False
for det_id in sorted(stats.keys()):
    n = stats[det_id]["n_frames"]
    if n == 0:
        continue
    p = pct(stats[det_id]["count_zero"], n)
    if p >= threshold:
        any_almost = True
        name = id_to_name.get(det_id, f"(id={det_id})")
        print(f"- ID={det_id} | Name={name} | frames={n} | count_zero={p:.1f}%")
if not any_almost:
    print("(none)")



### CONSTANT-ZERO CHECK (time-series) for LD-LSA16
Daily files read: 175
Selected detector names: ['D24', 'VD11', 'VD12', 'VD13', 'VD23', 'VD31', 'VD32', 'VD43', 'VD44', 'WD22']
Selected detector ids found: [1, 2, 4, 5, 6, 7, 10, 11, 12, 33]

=== Selected detectors: usability indicators ===
- ID=1 | Name=VD11 | frames=16714 | count_zero=601 (3.6%) | occ_zero=3760 (22.5%) | vel_zero=16714 (100.0%)
- ID=2 | Name=VD12 | frames=16714 | count_zero=530 (3.2%) | occ_zero=3634 (21.7%) | vel_zero=16714 (100.0%)
- ID=4 | Name=WD22 | frames=16714 | count_zero=2628 (15.7%) | occ_zero=3597 (21.5%) | vel_zero=16714 (100.0%)
- ID=5 | Name=VD23 | frames=16714 | count_zero=670 (4.0%) | occ_zero=2809 (16.8%) | vel_zero=16714 (100.0%)
- ID=6 | Name=VD31 | frames=16714 | count_zero=2667 (16.0%) | occ_zero=6919 (41.4%) | vel_zero=16714 (100.0%)
- ID=7 | Name=VD32 | frames=16714 | count_zero=887 (5.3%) | occ_zero=3643 (21.8%) | vel_zero=16714 (100.0%)
- ID=10 | Name=VD43 | frames=16714 | count_zero=589 (3.

In [13]:
print("\n==============================")
print("DETECTOR SELECTION SUMMARY")
print("==============================")

# Senin kullanmak istediğin isimler
print("\nDetectors you intended to use:")
for name in sorted(selected_names):
    print(" -", name)

usable_final = []
unusable_final = []

for name in selected_names:
    det_id = name_to_id.get(name)
    if det_id is None:
        unusable_final.append((name, "not found in definition"))
        continue

    n = stats[det_id]["n_frames"]
    cz = stats[det_id]["count_zero"]

    if n > 0 and cz == n:
        unusable_final.append((name, "count always 0"))
    else:
        usable_final.append(name)

print("\nUSABLE detectors (for calibration / LSTM):")
for u in sorted(usable_final):
    print(" ✅", u)

print("\nUNUSABLE detectors:")
for name, reason in unusable_final:
    print(f" ❌ {name}  -> {reason}")



DETECTOR SELECTION SUMMARY

Detectors you intended to use:
 - D24
 - VD11
 - VD12
 - VD13
 - VD23
 - VD31
 - VD32
 - VD43
 - VD44
 - WD22

USABLE detectors (for calibration / LSTM):
 ✅ VD11
 ✅ VD12
 ✅ VD13
 ✅ VD23
 ✅ VD31
 ✅ VD32
 ✅ VD43
 ✅ VD44
 ✅ WD22

UNUSABLE detectors:
 ❌ D24  -> count always 0


The upstream detector D24 was excluded from further analysis, as it produced zero vehicle counts in 100% of all observed time frames and is therefore considered non-functional.


**Detector Health accross the corridor**

The study corridor is structured into a main east–west spine (LD-LSA16, 10, 8, 1, 42, 40) and additional connector intersections (LD-LSA24, 29, 32, 3, 11, 7, 5, 4) that feed or divert traffic to/from the corridor.


✅ Reads all corridor intersections  
✅ Separates them into MAIN corridor and CONNECTOR branches  
✅ Analyzes the time-series detector files  
✅ Identifies detectors with constant-zero outputs  
✅ Flags controller-reported faulty detectors  
✅ Determines usability (usable / unusable) for AI and calibration  
✅ Generates a corridor_detector_health_report.csv file  
✅ Prints a grouped summary in the console  
The script processes all corridor intersections, classifies them into main and connector groups, evaluates historical detector time-series data, identifies constant-zero and controller-fault detectors, determines their suitability for calibration and AI modeling, generates a comprehensive detector health report (CSV), and provides a summarized group-level overview in the console.



In [14]:
import os
import json
import csv
from collections import defaultdict

# ==============================
# PATHS
# ==============================
intersections_folder = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"
landau_folder = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"

# ==============================
# CORRIDOR GROUPS
# ==============================
main_corridor = {"LD-LSA16", "LD-LSA10", "LD-LSA8", "LD-LSA1", "LD-LSA42", "LD-LSA40"}
connectors = {"LD-LSA24", "LD-LSA29", "LD-LSA32", "LD-LSA3", "LD-LSA11", "LD-LSA7", "LD-LSA5", "LD-LSA4"}

important_intersections = main_corridor.union(connectors)

def get_group(alias):
    if alias in main_corridor:
        return "MAIN_CORRIDOR_EW"
    if alias in connectors:
        return "CONNECTOR_BRANCH"
    return "OTHER"

# ==============================
# HELPERS
# ==============================
def find_json_files(directory):
    if not os.path.isdir(directory):
        return []
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith(".json")
    ]

def pct(z, n):
    return 0.0 if n == 0 else (100.0 * z / n)

def safe_get_reading(det_obj, key):
    return det_obj.get("reading", {}).get(key, {}).get("value")

# ==============================
# LOAD INTERSECTION DEFINITIONS
# ==============================
def load_definition_maps():
    definition_map = {}

    for fp in find_json_files(intersections_folder):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)

        tlc = data.get("tlc", {})
        alias = tlc.get("alias")

        if alias not in important_intersections:
            continue

        id_to_name = {}
        for sw in tlc.get("softwareStatus", []):
            for d in sw.get("node", {}).get("detectors", []):
                if d.get("id") is not None and d.get("name"):
                    id_to_name[int(d["id"])] = d["name"]

        hw = tlc.get("hardwareStatus", {})
        fault_list = hw.get("faultDetectors", []) or []
        controller_fault_names = {fd.get("name") for fd in fault_list if fd.get("name")}

        definition_map[alias] = {
            "id_to_name": id_to_name,
            "controller_fault_names": controller_fault_names,
            "meta": {
                "state": tlc.get("state"),
                "active": tlc.get("active"),
                "brokenDetectorCount": tlc.get("brokenDetectorCount"),
                "powerOk": hw.get("powerOk"),
                "emergencyOff": hw.get("emergencyOff"),
            }
        }

    return definition_map

# ==============================
# ANALYZE TIME SERIES
# ==============================
def analyze_timeseries(alias):
    ts_folder = os.path.join(landau_folder, alias)
    stats = defaultdict(lambda: {"n":0,"cz":0,"cnz":0})

    if not os.path.isdir(ts_folder):
        return stats, 0

    files = sorted(find_json_files(ts_folder))

    for fp in files:
        with open(fp, "r", encoding="utf-8") as f:
            day = json.load(f)

        for tf in day.get("timeFrames", []):
            for det in tf.get("detectors", []):
                det_id = det.get("id")
                if det_id is None:
                    continue
                det_id = int(det_id)

                stats[det_id]["n"] += 1
                count_val = safe_get_reading(det, "count")

                if isinstance(count_val, (int,float)):
                    if count_val == 0:
                        stats[det_id]["cz"] += 1
                    else:
                        stats[det_id]["cnz"] += 1

    return stats, len(files)

# ==============================
# MAIN RUN
# ==============================
definition_map = load_definition_maps()

rows = []
group_summary = defaultdict(lambda: {"usable":0,"unusable":0})

for alias in sorted(definition_map.keys()):

    stats, n_files = analyze_timeseries(alias)
    id_to_name = definition_map[alias]["id_to_name"]
    controller_fault = definition_map[alias]["controller_fault_names"]
    meta = definition_map[alias]["meta"]

    print(f"\n### {alias} ({get_group(alias)}) | days={n_files} | state={meta['state']}")

    for det_id, st in stats.items():
        n = st["n"]
        if n == 0:
            continue

        name = id_to_name.get(det_id, f"(id_{det_id})")
        zero_pct = pct(st["cz"], n)

        constant_zero = (st["cz"] == n and st["cnz"] == 0)
        fault = (name in controller_fault)

        usable = (not fault) and (not constant_zero)

        if usable:
            group_summary[get_group(alias)]["usable"] += 1
        else:
            group_summary[get_group(alias)]["unusable"] += 1

        rows.append({
            "group": get_group(alias),
            "intersection": alias,
            "detector_id": det_id,
            "detector_name": name,
            "frames": n,
            "count_zero_pct": round(zero_pct,2),
            "controller_fault": fault,
            "constant_zero": constant_zero,
            "usable_for_AI": usable,
            "state": meta["state"],
            "brokenDetectorCount": meta["brokenDetectorCount"]
        })

# ==============================
# SAVE CSV
# ==============================
output_path = os.path.join(os.getcwd(), "corridor_detector_health_report.csv")

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("\n=== GROUP SUMMARY ===")
for g in group_summary:
    print(f"{g}: usable={group_summary[g]['usable']} | unusable={group_summary[g]['unusable']}")

print("\nCSV saved to:", output_path)




### LD-LSA1 (MAIN_CORRIDOR_EW) | days=175 | state=INTERNAL_FAILURE

### LD-LSA10 (MAIN_CORRIDOR_EW) | days=175 | state=NO_FAILURE

### LD-LSA11 (CONNECTOR_BRANCH) | days=175 | state=NO_FAILURE

### LD-LSA16 (MAIN_CORRIDOR_EW) | days=175 | state=DETECTOR_FAILURE

### LD-LSA24 (CONNECTOR_BRANCH) | days=175 | state=NO_FAILURE

### LD-LSA29 (CONNECTOR_BRANCH) | days=175 | state=NO_FAILURE

### LD-LSA3 (CONNECTOR_BRANCH) | days=175 | state=NO_FAILURE

### LD-LSA32 (CONNECTOR_BRANCH) | days=175 | state=NO_FAILURE

### LD-LSA4 (CONNECTOR_BRANCH) | days=175 | state=INTERNAL_FAILURE

### LD-LSA40 (MAIN_CORRIDOR_EW) | days=175 | state=CONNECTION_ERROR

### LD-LSA42 (MAIN_CORRIDOR_EW) | days=175 | state=NO_FAILURE

### LD-LSA5 (CONNECTOR_BRANCH) | days=175 | state=INTERNAL_FAILURE

### LD-LSA7 (CONNECTOR_BRANCH) | days=0 | state=NO_FAILURE

### LD-LSA8 (MAIN_CORRIDOR_EW) | days=175 | state=ERROR

=== GROUP SUMMARY ===
MAIN_CORRIDOR_EW: usable=145 | unusable=15
CONNECTOR_BRANCH: usable=101 | unus

In [15]:
import os
import json
from collections import defaultdict

# ==============================
# PATHS
# ==============================
intersections_folder = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"
landau_folder = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"

main_corridor = {"LD-LSA16", "LD-LSA10", "LD-LSA8", "LD-LSA1", "LD-LSA42", "LD-LSA40"}
connectors = {"LD-LSA24", "LD-LSA29", "LD-LSA32", "LD-LSA3", "LD-LSA11", "LD-LSA7", "LD-LSA5", "LD-LSA4"}
important_intersections = main_corridor.union(connectors)

ALMOST_ZERO_THRESHOLD = 99.0

# ==============================
def find_json_files(directory):
    if not os.path.isdir(directory):
        return []
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith(".json")
    ]

def pct(z, n):
    return 0.0 if n == 0 else (100.0 * z / n)

def safe_get_count(det_obj):
    return det_obj.get("reading", {}).get("count", {}).get("value")

# ==============================
# LOAD DEFINITION MAPS
# ==============================
def load_definitions():
    def_map = {}

    for fp in find_json_files(intersections_folder):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)

        tlc = data.get("tlc", {})
        alias = tlc.get("alias")

        if alias not in important_intersections:
            continue

        id_to_name = {}
        for sw in tlc.get("softwareStatus", []):
            for d in sw.get("node", {}).get("detectors", []):
                if d.get("id") and d.get("name"):
                    id_to_name[int(d["id"])] = d["name"]

        hw = tlc.get("hardwareStatus", {})
        fault_list = hw.get("faultDetectors", []) or []
        fault_names = {fd.get("name") for fd in fault_list if fd.get("name")}

        def_map[alias] = {
            "id_to_name": id_to_name,
            "fault_names": fault_names,
            "state": tlc.get("state"),
            "brokenDetectorCount": tlc.get("brokenDetectorCount")
        }

    return def_map

# ==============================
# ANALYZE
# ==============================
definitions = load_definitions()

for alias in sorted(definitions.keys()):

    ts_folder = os.path.join(landau_folder, alias)
    if not os.path.isdir(ts_folder):
        continue

    stats = defaultdict(lambda: {"n":0,"zero":0,"nonzero":0})
    files = sorted(find_json_files(ts_folder))

    for fp in files:
        with open(fp, "r", encoding="utf-8") as f:
            day = json.load(f)

        for tf in day.get("timeFrames", []):
            for det in tf.get("detectors", []):
                det_id = det.get("id")
                if det_id is None:
                    continue

                det_id = int(det_id)
                stats[det_id]["n"] += 1

                c = safe_get_count(det)
                if isinstance(c, (int,float)):
                    if c == 0:
                        stats[det_id]["zero"] += 1
                    else:
                        stats[det_id]["nonzero"] += 1

    print("\n=====================================")
    print(f"INTERSECTION: {alias}")
    print("State:", definitions[alias]["state"],
          "| BrokenDetectors:", definitions[alias]["brokenDetectorCount"])

    id_to_name = definitions[alias]["id_to_name"]
    fault_names = definitions[alias]["fault_names"]

    for det_id, st in stats.items():
        n = st["n"]
        if n == 0:
            continue

        name = id_to_name.get(det_id, f"(id_{det_id})")
        zero_pct = pct(st["zero"], n)

        if st["zero"] == n:
            print(f"❌ {name} (ID={det_id}) → ALWAYS ZERO ({n} frames)")
        elif zero_pct >= ALMOST_ZERO_THRESHOLD:
            print(f"⚠️ {name} (ID={det_id}) → {zero_pct:.1f}% ZERO")

        if name in fault_names:
            print(f"🚨 {name} (ID={det_id}) → CONTROLLER FAULT REPORTED")



INTERSECTION: LD-LSA1
State: INTERNAL_FAILURE | BrokenDetectors: 2
❌ DET51 (ID=25) → ALWAYS ZERO (2787 frames)
🚨 DET51 (ID=25) → CONTROLLER FAULT REPORTED
❌ DET72 (ID=29) → ALWAYS ZERO (2787 frames)
🚨 DET72 (ID=29) → CONTROLLER FAULT REPORTED

INTERSECTION: LD-LSA10
State: NO_FAILURE | BrokenDetectors: 0
❌ PC (ID=34) → ALWAYS ZERO (12886 frames)
⚠️ GB (ID=35) → 99.3% ZERO
⚠️ FR (ID=36) → 100.0% ZERO
❌ WBK1_Err (ID=38) → ALWAYS ZERO (12886 frames)
❌ WBK2_Err (ID=39) → ALWAYS ZERO (12886 frames)
❌ WBK3_Err (ID=40) → ALWAYS ZERO (12886 frames)
⚠️ VKx_iO (ID=41) → 100.0% ZERO

INTERSECTION: LD-LSA11
State: NO_FAILURE | BrokenDetectors: 0

INTERSECTION: LD-LSA16
State: DETECTOR_FAILURE | BrokenDetectors: 2
🚨 WD41 (ID=8) → CONTROLLER FAULT REPORTED
🚨 WD42 (ID=9) → CONTROLLER FAULT REPORTED
❌ D24 (ID=12) → ALWAYS ZERO (16714 frames)
⚠️ WBK1_Err (ID=22) → 99.6% ZERO
⚠️ WBK2_Err (ID=23) → 99.6% ZERO
❌ VK14_i_O (ID=24) → ALWAYS ZERO (16714 frames)
❌ PC (ID=27) → ALWAYS ZERO (16714 frames)
❌ GB 